# 3 — Split Kaggle (validation/test) temporale

Input: `data/processed/kaggle_hf_like_from_2025-03-25.jsonl`

Output:
- `data/processed/kaggle_val.jsonl`
- `data/processed/kaggle_test.jsonl`

Split temporale per date.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

kaggle_jsonl = PROC_DIR / 'kaggle_hf_like_from_2025-03-25.jsonl'
kaggle_jsonl

WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/kaggle_hf_like_from_2025-03-25.jsonl')

In [2]:
import sys
sys.path.append(str((PROJECT_ROOT / 'src').resolve()))

from data.jsonl_io import read_jsonl, write_jsonl
from data.splits import split_by_fraction, split_by_date, stats
from utils.hashing import puzzle_hash


In [3]:
records = read_jsonl(kaggle_jsonl)
records
print('Loaded records:', len(records))
print('Overall stats:', stats(records))

Loaded records: 264
Overall stats: {'num_records': 264, 'num_unique_dates': 264, 'min_date': '2025-03-25', 'max_date': '2025-12-15'}


## Parametri split

- Se `USE_FRACTION=True`: il test prende le date più recenti secondo `TEST_FRAC`.
- Se `USE_FRACTION=False`: `SPLIT_DATE` è la prima data che va nel test (>=).

In [4]:
USE_FRACTION = True
TEST_FRAC = 0.5

SPLIT_DATE = '2025-06-01'


In [5]:
if USE_FRACTION:
    val, test, used_split_date = split_by_fraction(records, test_frac=TEST_FRAC)
else:
    val, test = split_by_date(records, split_date=SPLIT_DATE)
    used_split_date = SPLIT_DATE

print('Used split_date:', used_split_date)
print('VAL stats:', stats(val))
print('TEST stats:', stats(test))

Used split_date: 2025-08-05
VAL stats: {'num_records': 132, 'num_unique_dates': 132, 'min_date': '2025-03-25', 'max_date': '2025-08-04'}
TEST stats: {'num_records': 132, 'num_unique_dates': 132, 'min_date': '2025-08-05', 'max_date': '2025-12-15'}


In [6]:
val_h = {puzzle_hash(r['words']) for r in val}
test_h = {puzzle_hash(r['words']) for r in test}
print('Hash overlap val/test:', len(val_h & test_h))

Hash overlap val/test: 0


In [7]:
val_path = PROC_DIR / 'kaggle_val.jsonl'
test_path = PROC_DIR / 'kaggle_test.jsonl'
write_jsonl(val, val_path)
write_jsonl(test, test_path)
print('Saved:', val_path)
print('Saved:', test_path)

Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\kaggle_val.jsonl
Saved: C:\Users\cola0\Desktop\nlp.project-Colangelo-2526\data\processed\kaggle_test.jsonl
